In [6]:
from dotenv import load_dotenv
from agents import Agent, Runner, OpenAIChatCompletionsModel, trace, function_tool
from openai import AsyncOpenAI
import asyncio
import os
import feedparser
import requests
from datetime import datetime, timedelta
from typing import List, Dict, Optional
from bs4 import BeautifulSoup
import time


In [ ]:
# The usual starting point

load_dotenv(override=True)

True

In [9]:
current_date = datetime.now().strftime("%Y-%m-%d")

In [12]:
news_collector_instructions = f"""You are a news aggregator. Your job is to find the top 5 most popular news stories in the world today. Evaluation of popularity is at your discretion. 
The current date is {{current_date}}. 
You should return a list of the 5 articles you find along with the source, title, and a short summary of the article. This should be returned in JSON format of the following form only:

[
    {{
        "source": "source",
        "title": "title",
        "summary": "summary"
    }}
]"""

In [22]:
news_aggregator = Agent(
        name="News Aggregator",
        instructions=news_collector_instructions,
        model="gpt-5-mini"
)

In [23]:
message = "Please collect today's news."

with trace("News collection"):
    results = await asyncio.gather(
        Runner.run(news_aggregator, message)
    )

print(results[0].final_output + "\n\n")


[
    {
        "source": "Assistant — no live web access",
        "title": "Unable to fetch live news for today",
        "summary": "I don't have browsing access to collect today's live news. I can fetch articles if you provide URLs, or I can summarize news you paste here, or suggest how to connect me to a news API or upload a news feed."
    },
    {
        "source": "Assistant — no live web access",
        "title": "Can’t access current headlines without external input",
        "summary": "I cannot access the internet or real-time news. If you want current headlines, please paste links or tell me which news sources or topics (e.g., world, finance, tech) to prioritize, and I will summarize the provided content."
    },
    {
        "source": "Assistant — no live web access",
        "title": "Options to get today’s news from me",
        "summary": "Options: (1) paste article links for summarization, (2) upload a news feed or articles, (3) grant access to a news API endpoint (y

In [1]:
# Import the NewsCollector class
from news_collector import NewsCollector

In [2]:
# Create a NewsCollector instance and test the RSS feed
collector = NewsCollector(
    rss_url="https://news.google.com/rss?hl=en-US&gl=US&ceid=US:en",
    max_age_hours=24
)

print("Testing RSS feed fetch...")
articles = collector.fetch_news(num_results=10)
print(f"\nSuccessfully fetched {len(articles)} articles from RSS feed")

Testing RSS feed fetch...

Successfully fetched 10 articles from RSS feed


In [3]:
# Display the fetched articles
if articles:
    print(f"\n{'='*80}")
    print(f"RSS Feed Test Results - {len(articles)} articles found")
    print(f"{'='*80}\n")
    
    for i, article in enumerate(articles[:5], 1):
        print(f"{i}. {article['title']}")
        print(f"   Source: {article['source']}")
        print(f"   Published: {article['published']}")
        if article['summary']:
            summary = article['summary'][:150] + "..." if len(article['summary']) > 150 else article['summary']
            print(f"   Summary: {summary}")
        print(f"   Link: {article['link']}")
        print()
else:
    print("No articles were fetched. Check the RSS feed URL and network connection.")


RSS Feed Test Results - 10 articles found

1. Russian hits Ukraine energy sites in 'most powerful blow" so far this year - BBC
   Source: CBMiWkFVX3lxTE9qSmRSTkNIY2JkaE0zQzFWejNuckRRdENTbXRVN0ROeG5Ld1VOV09zRVpmeThjOUpzNnBzNldlVEJvSDU3dUhyaGtnOUd4dXBGQS1aUGJKbGlOUQ?oc=5
   Published: 2026-02-03T13:17:53
   Summary: <ol><li><a href="https://news.google.com/rss/articles/CBMiWkFVX3lxTE9qSmRSTkNIY2JkaE0zQzFWejNuckRRdENTbXRVN0ROeG5Ld1VOV09zRVpmeThjOUpzNnBzNldlVEJvSDU3...
   Link: https://news.google.com/rss/articles/CBMiWkFVX3lxTE9qSmRSTkNIY2JkaE0zQzFWejNuckRRdENTbXRVN0ROeG5Ld1VOV09zRVpmeThjOUpzNnBzNldlVEJvSDU3dUhyaGtnOUd4dXBGQS1aUGJKbGlOUQ?oc=5

2. Investigation Finds Credit Suisse Had Wider Nazi Ties Than Previously Known - The New York Times
   Source: CBMihAFBVV95cUxQMGUtM2RXa3lDNkhBcHl0SDVmcHlpT2t0N3ZKVkNzVnMyVVZVaUI4aTQ1LTRETjN1cHJBNWFsUzVRVXd5OVBfa2QwRVZNYTZiYURHak5rQW01alFrMlFMMW5mUktqNmM0eFhYX3hETWt3MHJRM1plODlydmc3YnVra1BwVUY?oc=5
   Published: 2026-02-03T11:00:23
   Summary: <ol>

In [5]:
# Test with a specific query
print("Testing RSS feed with a query parameter...")
query_articles = collector.fetch_news(query="stock%20market", num_results=5)
print(f"\nFound {len(query_articles)} articles for query 'stock market'")

if query_articles:
    for i, article in enumerate(query_articles, 1):
        print(f"\n{i}. {article['title']}")
        print(f"   Source: {article['source']}")
        print(f"   Published: {article['published']}")

Testing RSS feed with a query parameter...

Found 4 articles for query 'stock market'

1. Stock market today: Dow, S&P 500, Nasdaq whipsaw higher as gold, bitcoin see big swings amid earnings flood - Yahoo Finance
   Source: CBMi6AFBVV95cUxNTEVaYkZXbjZsam9ZclRGT25EUVZ0SVNmT0hvRUZnNnlHNmVFWDQ5ZkJnQWxhSDBBTDdQTS1VNlhJVWtvOGVsY3gyanpDVFlQZkl0eTBOWmJyS2ROZ25xZmtOSThzaWk1aUJqdXhfOTNBRmNSNXpjOXE2ZVcxeG5sUTBCcFIzZ2hnWlpfdl9aSHNEaWlSOGdsMThVQ3ZOdzcwWmxYWUxlc0ljcnUxZXc1Ny12WmJzYWpTQUhjenRNaEdQbzNkRk5DT0RGWDU1UDlyT0tMTkdISVpkczZ4Zl8xS2dZRjRmUF9l?oc=5
   Published: 2026-02-02T20:25:29

2. Stock Market Today: Dow Inches Higher As Palantir Rallies; Rambus Tumbles (Live Coverage) - Investor's Business Daily
   Source: CBMivwFBVV95cUxQaHBlRlVMRjNaelpqM3pVOVJCb2J0UWN6SGRIQUUyZ2ZyeW5icTRxWFJmSkl4eEYwT0NlaUZUay1ZcG8tcVo1X3JvUXdlOVdtMFlkSjEwZE0tNTB2ekdiSUowdVN4WkY0NWtyNHJQU0p2SUdhaWhYdHQzcmRhZ3F3alNESVM0RTdEWW9paU9KSERVQVdVWVBnOUlNS0Y4d005aTNieWNRUzZpNzRnUjhrTUZ5cExsVG1WSzV6eWM0OA?oc=5
   Published: 2026

In [7]:
# Create a NewsCollector instance that will be used by the tool
collector = NewsCollector(
    rss_url="https://news.google.com/rss?hl=en-US&gl=US&ceid=US:en",
    max_age_hours=24
)

# Create a tool function for fetching news
@function_tool
def fetch_news_from_rss(query: Optional[str] = None, num_results: int = 20) -> List[Dict]:
    """
    Fetch news articles from Google News RSS feed.
    
    Args:
        query: Optional search query to filter news (e.g., "stock market", "technology"). 
               If None, fetches general top news stories.
        num_results: Maximum number of articles to fetch (default: 20, max recommended: 50)
    
    Returns:
        List of news article dictionaries, each containing:
        - title: Article headline
        - source: News source name
        - link: URL to the full article
        - published: Publication date in ISO format
        - summary: Article summary or excerpt
    """
    articles = collector.fetch_news(query=query, num_results=num_results)
    return articles

In [10]:
# Update the news aggregator instructions to use the tool
news_collector_instructions_with_tool = f"""You are a news aggregator. Your job is to find the top 5 most popular news stories in the world today. 
The current date is {current_date}. 

You have access to a tool called fetch_news_from_rss that can retrieve real-time news from Google News RSS feeds. 
You MUST use this tool to fetch actual news articles. You can:
- Call it without a query to get general top news stories
- Call it with a query parameter to search for specific topics (e.g., "stock market", "technology", "politics")
- Specify num_results to control how many articles to fetch (recommended: 10-20 to get a good selection)

After fetching the news using the tool, analyze the articles and select the top 5 most popular/important stories based on:
- Relevance and impact
- Recency (prefer more recent articles)
- Source credibility
- Global significance

You should return a list of the 5 articles you find along with the source, title, and a short summary of the article. This should be returned in JSON format of the following form only:

[
    {{
        "source": "source",
        "title": "title",
        "summary": "summary"
    }}
]

Remember: Always use the fetch_news_from_rss tool to get real news data. Never make up or hallucinate news stories."""

In [11]:
# Create the news aggregator agent with the tool
news_aggregator_with_tool = Agent(
    name="News Aggregator",
    instructions=news_collector_instructions_with_tool,
    tools=[fetch_news_from_rss],
    model="gpt-5-mini"
)

In [12]:
# Test the news aggregator with the tool
message = "Please collect today's news."

with trace("News collection with tool"):
    result = await Runner.run(news_aggregator_with_tool, message)

print(result.final_output + "\n\n")

[
    {
        "source": "BBC",
        "title": "Russian hits Ukraine energy sites in 'most powerful blow\" so far this year - BBC",
        "summary": "Russian forces carried out heavy strikes on Ukraine’s energy infrastructure — described by sources as the most powerful attack so far this year — damaging power facilities and causing widespread outages as the conflict continues."
    },
    {
        "source": "The Washington Post",
        "title": "Satellite imagery shows where the U.S. military is positioned near Iran - The Washington Post",
        "summary": "Satellite images and reporting map recent U.S. military deployments in the region near Iran as tensions rise; the coverage outlines where forces and carriers are positioned amid concerns about potential escalation."
    },
    {
        "source": "BBC",
        "title": "X offices raided in France as UK opens fresh investigation into Grok - BBC",
        "summary": "French prosecutors' cybercrime unit searched X's Paris of